# 🔬 Step 2 — YOLOv8 Training on Balanced Dataset
### 150 Epochs | Balanced AP & MP | Loss Curves | Confusion Matrix | Metrics Table | best.pt
> ⚠️ Run STEP1_balance_and_split notebook FIRST before running this

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Install
# ═══════════════════════════════════════════════════════════
import os
os.system('pip install ultralytics -q')
print('✅ ultralytics installed')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Imports
# ═══════════════════════════════════════════════════════════
import sys, yaml, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
from ultralytics import YOLO

warnings.filterwarnings('ignore')
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — enable GPU in Settings!"}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Config
# ═══════════════════════════════════════════════════════════
# ← Uses balanced dataset yaml from Step 1
data_yaml_path = '/kaggle/input/notebooks/rugvedkulkarni2509/dataset-spliting/data_balanced.yaml'

OUTPUT_DIR    = '/kaggle/working/esophageal_yolov8'
EPOCHS        = 150
IMG_SIZE      = 640
BATCH_SIZE    = 16        # lower to 8 if OOM error
MODEL_VARIANT = 'yolov8m.pt'
PATIENCE      = 75        # half of 150
DEVICE        = 0 if torch.cuda.is_available() else 'cpu'
CLASS_NAMES   = ['AP', 'MP']

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify yaml exists
assert Path(data_yaml_path).exists(), f'❌ data_balanced.yaml not found! Run Step 1 first.'

print('✅ Config OK')
print(f'   data.yaml  : {data_yaml_path}')
print(f'   Device     : {DEVICE}')
print(f'   Classes    : {CLASS_NAMES}')
print(f'   Epochs     : {EPOCHS}')
print(f'   Patience   : {PATIENCE}')
print(f'   Batch size : {BATCH_SIZE}')
print(f'   Workers    : 4')
print()
print('data.yaml contents:')
print(open(data_yaml_path).read())

In [ ]:
# ═══════════════════════════════════════════════════════════/kaggle/input/notebooks/rugvedkulkarni2509/dataset-spliting/balanced_dataset
# CELL 4 — Verify balanced dataset image counts
# ═══════════════════════════════════════════════════════════
balanced_root = '/kaggle/input/notebooks/rugvedkulkarni2509/dataset-spliting/balanced_dataset'
print('Balanced dataset verification:')
print(f'{"Split":<8} {"Images":>8} {"Labels":>8}')
print('-'*28)
for split in ['train', 'valid', 'test']:
    imgs = len(list((Path(balanced_root)/split/'images').glob('*.*')))
    lbls = len(list((Path(balanced_root)/split/'labels').glob('*.txt')))
    ok   = '✅' if imgs == lbls else '⚠️'
    print(f'{ok} {split:<6} {imgs:>8} {lbls:>8}')

In [ ]:
# Fix yaml path — rewrite with correct input path
import yaml

correct_base = '/kaggle/input/notebooks/rugvedkulkarni2509/dataset-spliting/balanced_dataset'

cfg = {
    'path'  : correct_base,
    'train' : 'train/images',
    'val'   : 'valid/images',
    'test'  : 'test/images',
    'nc'    : 2,
    'names' : ['AP', 'MP'],
}

fixed_yaml = '/kaggle/working/data_fixed.yaml'
with open(fixed_yaml, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

data_yaml_path = fixed_yaml   # override the variable

print('✅ Fixed yaml written')
print(open(fixed_yaml).read())

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — TRAIN 150 EPOCHS on Balanced Dataset
# ═══════════════════════════════════════════════════════════
model = YOLO(MODEL_VARIANT)

results = model.train(
    data        = data_yaml_path,
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH_SIZE,
    workers     = 4,
    device      = DEVICE,
    patience    = PATIENCE,
    project     = OUTPUT_DIR,
    name        = 'train',
    save        = True,
    save_period = 25,
    plots       = True,
    val         = True,
    verbose     = True,
    exist_ok    = True,
    # Augmentation
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    degrees     = 10.0,
    translate   = 0.1,
    scale       = 0.5,
    shear       = 2.0,
    flipud      = 0.3,
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.1,
)

run_dir  = Path(OUTPUT_DIR) / 'train'
best_pt  = run_dir / 'weights' / 'best.pt'
csv_path = run_dir / 'results.csv'
print(f'\n✅ Training complete!')
print(f'   best.pt → {best_pt}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Loss Curves + mAP Curves
# ═══════════════════════════════════════════════════════════
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
print('CSV columns:', df.columns.tolist())

train_loss_cols = [c for c in df.columns if 'train' in c.lower() and 'loss' in c.lower()]
val_loss_cols   = [c for c in df.columns if 'val'   in c.lower() and 'loss' in c.lower()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('YOLOv8 — Esophageal Cancer — Loss Curves (150 Epochs, Balanced)', fontsize=14, fontweight='bold')

for ax, (key, title) in zip(axes, [('box_loss','Box Loss'),('cls_loss','Class Loss'),('dfl_loss','DFL Loss')]):
    t_col = next((c for c in train_loss_cols if key in c.lower()), None)
    v_col = next((c for c in val_loss_cols   if key in c.lower()), None)
    if t_col:
        ax.plot(df.index+1, df[t_col], label='Train', color='#2196F3', linewidth=1.8)
    if v_col:
        ax.plot(df.index+1, df[v_col], label='Val',   color='#F44336', linewidth=1.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_xlim(1, len(df))

plt.tight_layout()
loss_path = os.path.join(OUTPUT_DIR, 'loss_curves.png')
plt.savefig(loss_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Loss curves saved: {loss_path}')

map_cols = [c for c in df.columns if 'map' in c.lower()]
if map_cols:
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    colors = ['#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']
    for i, col in enumerate(map_cols):
        ax2.plot(df.index+1, df[col], label=col.strip(), color=colors[i%len(colors)], linewidth=1.8)
    ax2.set_title('mAP over Epochs (150 Epochs, Balanced)', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('mAP')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    ax2.set_xlim(1, len(df))
    plt.tight_layout()
    map_path = os.path.join(OUTPUT_DIR, 'map_curves.png')
    plt.savefig(map_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ mAP curves saved: {map_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Validate with best.pt
# ═══════════════════════════════════════════════════════════
best_model  = YOLO(str(best_pt))
val_results = best_model.val(
    data     = data_yaml_path,
    imgsz    = IMG_SIZE,
    batch    = BATCH_SIZE,
    device   = DEVICE,
    conf     = 0.25,
    iou      = 0.5,
    plots    = True,
    save_json= True,
    project  = OUTPUT_DIR,
    name     = 'val_best',
    exist_ok = True,
    verbose  = True,
)
print('✅ Validation complete')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Confusion Matrix
# ═══════════════════════════════════════════════════════════
cm_array = None
try:
    cm_array = val_results.confusion_matrix.matrix
    print('✅ Confusion matrix loaded from val_results')
except Exception:
    pass

if cm_array is None:
    npy = Path(OUTPUT_DIR) / 'val_best' / 'confusion_matrix.npy'
    if npy.exists():
        cm_array = np.load(str(npy))
        print(f'✅ Loaded from {npy}')

assert cm_array is not None, '❌ Could not load confusion matrix!'

nc = len(CLASS_NAMES)
if cm_array.shape[0] == nc + 1:
    cm_array = cm_array[:nc, :nc]

print(f'Shape: {cm_array.shape}')
print(cm_array)

cm_norm = cm_array.astype(float)
row_sums = cm_norm.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
cm_norm /= row_sums

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Confusion Matrix — Esophageal Cancer (AP vs MP) — Balanced', fontsize=13, fontweight='bold')

for ax, data, title, fmt in zip(
    axes,
    [cm_array.astype(int), cm_norm],
    ['Raw Counts', 'Normalised (Recall)'],
    ['d', '.2f'],
):
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap='Blues',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        ax=ax, linewidths=0.8, linecolor='gray',
        annot_kws={'size': 13, 'weight': 'bold'},
    )
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)

plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix_custom.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Confusion matrix saved: {cm_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Metrics Table (Precision / Recall / F1 / Accuracy)
# ═══════════════════════════════════════════════════════════
def compute_metrics(cm):
    rows  = []
    total = cm.sum()
    for i, cls in enumerate(CLASS_NAMES[:cm.shape[0]]):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = total - TP - FP - FN
        P   = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        R   = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        F1  = 2*P*R / (P + R) if (P + R)   > 0 else 0.0
        Acc = (TP + TN) / total if total    > 0 else 0.0
        rows.append({'Class': cls,
                     'TP': int(TP), 'FP': int(FP), 'FN': int(FN), 'TN': int(TN),
                     'Precision': round(P,4), 'Recall': round(R,4),
                     'F1-Score' : round(F1,4), 'Accuracy': round(Acc,4)})
    rows.append({'Class':'MACRO AVG', 'TP':'','FP':'','FN':'','TN':'',
                 'Precision': round(np.mean([r['Precision'] for r in rows]),4),
                 'Recall'   : round(np.mean([r['Recall']    for r in rows]),4),
                 'F1-Score' : round(np.mean([r['F1-Score']  for r in rows]),4),
                 'Accuracy' : round(np.mean([r['Accuracy']  for r in rows]),4)})
    return pd.DataFrame(rows)

metrics_df = compute_metrics(cm_array.astype(float))
print('\n' + '='*65)
print('  PER-CLASS METRICS')
print('='*65)
print(metrics_df.to_string(index=False))
print('='*65)

fig, ax = plt.subplots(figsize=(15, max(3, len(metrics_df)*0.65+1.2)))
ax.axis('off')
tbl = ax.table(cellText=metrics_df.values.tolist(),
               colLabels=list(metrics_df.columns),
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.2, 1.8)
for j in range(len(metrics_df.columns)):
    tbl[0, j].set_facecolor('#1565C0')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(metrics_df)):
    for j in range(len(metrics_df.columns)):
        tbl[i, j].set_facecolor('#F5F5F5' if i%2==0 else 'white')
last = len(metrics_df)
for j in range(len(metrics_df.columns)):
    tbl[last, j].set_facecolor('#BBDEFB')
    tbl[last, j].set_text_props(fontweight='bold')
ax.set_title('Esophageal Cancer YOLOv8 — Precision / Recall / F1 / Accuracy',
             fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
tbl_path = os.path.join(OUTPUT_DIR, 'metrics_table.png')
plt.savefig(tbl_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Metrics table saved: {tbl_path}')
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'metrics_table.csv'), index=False)
print(f'✅ Metrics CSV saved  : {os.path.join(OUTPUT_DIR, "metrics_table.csv")}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — Final Summary
# ═══════════════════════════════════════════════════════════
print('\n' + '='*65)
print('  ALL OUTPUT FILES')
print('='*65)
files = [
    ('Best Model Weights' , str(best_pt)),
    ('Loss Curves'        , os.path.join(OUTPUT_DIR, 'loss_curves.png')),
    ('mAP Curves'         , os.path.join(OUTPUT_DIR, 'map_curves.png')),
    ('Confusion Matrix'   , os.path.join(OUTPUT_DIR, 'confusion_matrix_custom.png')),
    ('Metrics Table PNG'  , os.path.join(OUTPUT_DIR, 'metrics_table.png')),
    ('Metrics CSV'        , os.path.join(OUTPUT_DIR, 'metrics_table.csv')),
    ('Before Balance Chart', '/kaggle/working/distribution_before.png'),
    ('After Balance Chart' , '/kaggle/working/distribution_after.png'),
]
for name, path in files:
    mark = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {mark}  {name:<24} → {path}')
print('\n🎉 Done! Download best.pt from the Output tab.')

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if 'data_balanced' in file or 'yaml' in file:
            print(os.path.join(root, file))

In [ ]:
# Check for duplicate filenames between train and test
from pathlib import Path

train_files = set(p.name for p in Path('/kaggle/input/notebooks/rugvedkulkarni2509/dataset-spliting/balanced_dataset/train/images').glob('*.*'))
test_files  = set(p.name for p in Path('/kaggle/input/notebooks/rugvedkulkarni2509/dataset-spliting/balanced_dataset/test/images').glob('*.*'))
val_files   = set(p.name for p in Path('/kaggle/input/notebooks/rugvedkulkarni2509/dataset-spliting/balanced_dataset/valid/images').glob('*.*'))

print(f'Train images : {len(train_files)}')
print(f'Val images   : {len(val_files)}')
print(f'Test images  : {len(test_files)}')
print()

# Check overlaps
train_test = train_files & test_files
train_val  = train_files & val_files
val_test   = val_files   & test_files

print(f'Train ∩ Test  overlap : {len(train_test)}')
print(f'Train ∩ Val   overlap : {len(train_val)}')
print(f'Val   ∩ Test  overlap : {len(val_test)}')
print()

if train_test or train_val or val_test:
    print('⚠️  DATA LEAKAGE DETECTED!')
else:
    print('✅ No overlap — data split is clean')
    print('→ 99% accuracy is likely due to dataset being too easy or augmented duplicates')